# 基于不同存储结构的求和算子实现——实验实操

完整实验已经展示了顺序存储与链式存储求和算子从数据准备、Ascend C Kernel、Host 调用到 NPU 运行验证的完整过程。本实践是在完整实验的基础上进行代码补全，学生可根据注释补全 `TODO`自行完成。

完成后应能够：

1. 使用 C++ 定义顺序存储和链式存储所需的数据结构；
2. 根据给定的物理布局构建两种存储结构；
3. 分别实现顺序遍历、链式遍历、求和以及链式访问顺序记录；
4. 将 C++ 中的访问逻辑迁移到 Ascend C Kernel，并在 NPU 上验证计算结果。


## 0. 实践介绍

### 0.1 实践任务

本实践共包含 4 个任务：

- **任务一：定义数据结构** —— 使用标准 C++ 定义顺序表和静态链表；
- **任务二：构建数据结构** —— 根据给定的输入数据构造两种存储结构；
- **任务三：实现求和算子** —— 分别基于两种存储结构实现求和算子，并体会链式存储结构实际遍历顺序；
- **任务四：Ascend C实现求和算子** —— 体会在昇腾NPU上实现求和算子核函数。

操作方式：

1. 先阅读任务说明；
2. 在带有 `TODO` 的代码单元中补全代码；
3. **重新运行该代码单元**，将修改后的 C++/Ascend C 代码写入文件；
4. 再运行对应的测试单元；
5. 测试通过后再进入下一项任务。

注：测试代码和工程辅助代码已经提供，不需要修改。本实践中的代码单元需要从上至下依次运行，否则会导致编译失败。


In [ ]:
from pathlib import Path
import subprocess
import os

PRACTICE_DIR = Path("Source/01.02").resolve()
(PRACTICE_DIR / "cpp").mkdir(parents=True, exist_ok=True)
(PRACTICE_DIR / "ascend_ops" / "op_kernel").mkdir(parents=True, exist_ok=True)
(PRACTICE_DIR / "ascend_ops" / "host_launch").mkdir(parents=True, exist_ok=True)
(PRACTICE_DIR / "scripts").mkdir(parents=True, exist_ok=True)

print("Practice directory:", PRACTICE_DIR)


### 0.2 输入数据

本实践只使用 4 个float类型的元素，便于直接观察逻辑顺序和物理访问顺序：

```text
逻辑数据：
10 → 20 → 30 → 40
```

顺序存储中，逻辑顺序与物理下标顺序一致。

链式存储中，为了体现物理位置不连续的特点，本实践**固定给定**逻辑位置到物理位置的映射为：

```text
逻辑顺序下标 : 0  1  2  3
物理位置下标: 0  2  3  1
```

也就是说，逻辑上的第 0、1、2、3 个元素分别存放在物理位置 0、2、3、1。

注：本实践不要求实现随机打乱。物理布局已经给定，学习重点是如何根据该布局建立链式关系，以及两种布局如何被遍历。


---
## 1. 任务一：定义数据结构

请定义下面 3 个数据结构：

- `ArrayList`：表示顺序表，只需要保存连续的数据数组；
- `ListNode`：表示单链表的一个结点，需要保存当前数据和后继结点的物理下标；
- `IndexList`：表示静态链表，需要保存所有结点以及头结点的物理下标。

提示：

- 可以使用 `std::vector<float>` 保存连续数据；
- `nextIndex == -1` 表示链尾；
- `first` 表示链表遍历的起点。

请只补全 `TODO` 部分。


In [ ]:
%%writefile Source/01.02/cpp/practice_types.hpp
#pragma once

#include <vector>

struct ArrayList {
    // TODO 1A：
    // 定义一个成员，用于保存顺序存储的 float 数据。
};

struct ListNode {
    // TODO 1B：
    // 定义当前结点保存的数据。
    // 定义后继结点的物理下标，链尾使用 -1。
};

struct IndexList {
    // TODO 1C：
    // 定义保存全部 ListNode 的容器。
    // 定义头结点的物理下标 first。
};


In [ ]:
%%writefile Source/01.02/cpp/task1_test.cpp
#include "practice_types.hpp"

#include <cstdint>
#include <iostream>
#include <type_traits>
#include <vector>

int main()
{
    static_assert(std::is_same_v<decltype(ArrayList{}.data), std::vector<float>>,
                  "ArrayList::data should be std::vector<float>");
    static_assert(std::is_same_v<decltype(ListNode{}.data), float>,
                  "ListNode::data should be float");
    static_assert(std::is_same_v<decltype(ListNode{}.nextIndex), int>,
                  "ListNode::nextIndex should be int");
    static_assert(std::is_same_v<decltype(IndexList{}.nodes), std::vector<ListNode>>,
                  "IndexList::nodes should be std::vector<ListNode>");
    static_assert(std::is_same_v<decltype(IndexList{}.first), int>,
                  "IndexList::first should be int");

    std::cout << "Task 1 PASSED: data structures are valid." << std::endl;
    return 0;
}


In [ ]:
from pathlib import Path
import subprocess

result = subprocess.run(
    [
        "g++",
        "-std=c++17",
        "-I", "Source/01.02/cpp",
        "Source/01.02/cpp/task1_test.cpp",
        "-o", "Source/01.02/cpp/task1_test",
    ],
    text=True,
    capture_output=True,
)

if result.returncode != 0:
    print("Task 1 编译未通过，请检查 TODO 部分：")
    print(result.stderr)
else:
    subprocess.run(
        ["Source/01.02/cpp/task1_test"],
        check=True
    )

---
## 2. 任务二：构建数据结构

现在使用同一组逻辑数据：

```cpp
{10.0f, 20.0f, 30.0f, 40.0f}
```

分别构造 `ArrayList` 和 `IndexList`。

### 2.1 顺序存储

顺序存储不改变数据的排列方式，逻辑数据按顺序放入连续数组即可。

### 2.2 链式存储

教师已经给出固定映射：

```cpp
logicalToPhysical = {0, 2, 3, 1};
```

其中：

```text
logicalToPhysical[i]
```

表示“逻辑上的第 `i` 个元素实际存放在哪个物理下标”。

构造链表时需要完成两件事：

1. 把 `input[i]` 放到对应物理结点；
2. 当前逻辑元素的 `nextIndex` 指向下一个逻辑元素所在的物理位置，最后一个结点的 `nextIndex` 为 `-1`。

注：固定映射已经写好，不需要实现打乱算法。


In [ ]:
%%writefile Source/01.02/cpp/practice_build.hpp
#pragma once

#include "practice_types.hpp"

#include <vector>

ArrayList buildArrayList(const std::vector<float>& input);
IndexList buildIndexList(const std::vector<float>& input);


In [ ]:
%%writefile Source/01.02/cpp/practice_build.cpp
#include "practice_build.hpp"

#include <stdexcept>
#include <vector>

ArrayList buildArrayList(const std::vector<float>& input)
{
    ArrayList result;

    // TODO 2A：
    // 将 input 中的逻辑数据按原顺序写入 ArrayList。
    // 提示：顺序存储不需要改变数据排列。

    return result;
}

IndexList buildIndexList(const std::vector<float>& input)
{
    if (input.size() != 4) {
        throw std::invalid_argument("this practice uses exactly 4 elements");
    }

    // 固定映射已经提供，不需要学生实现随机打乱。
    // logicalToPhysical[i] 表示逻辑位置 i 对应的物理位置。
    const std::vector<int> logicalToPhysical = {0, 2, 3, 1};

    IndexList result;
    result.nodes.resize(input.size());

    // TODO 2B：
    // 1. 按 logicalToPhysical 将 input 中的数据写入对应物理结点；
    // 2. 为每个结点设置 nextIndex：
    //    - 指向下一个逻辑元素所在的物理位置；
    //    - 最后一个逻辑元素的 nextIndex 设为 -1；
    // 3. 设置 result.first，使其指向逻辑上的第 0 个元素。

    return result;
}


In [ ]:
%%writefile Source/01.02/cpp/task2_test.cpp
#include "practice_build.hpp"

#include <cassert>
#include <iostream>
#include <vector>

int main()
{
    const std::vector<float> input = {10.0f, 20.0f, 30.0f, 40.0f};

    const ArrayList arrayList = buildArrayList(input);
    assert(arrayList.data == input);

    const IndexList indexList = buildIndexList(input);
    assert(indexList.nodes.size() == 4);
    assert(indexList.first == 0);

    const std::vector<float> expectedData = {10.0f, 40.0f, 20.0f, 30.0f};
    const std::vector<int> expectedNext = {2, -1, 3, 1};

    for (std::size_t i = 0; i < indexList.nodes.size(); ++i) {
        assert(indexList.nodes[i].data == expectedData[i]);
        assert(indexList.nodes[i].nextIndex == expectedNext[i]);
    }

    std::cout << "ArrayList data: ";
    for (float v : arrayList.data) std::cout << v << ' ';

    std::cout << "\nIndexList physical data: ";
    for (const auto& node : indexList.nodes) std::cout << node.data << ' ';

    std::cout << "\nIndexList nextIndex: ";
    for (const auto& node : indexList.nodes) std::cout << node.nextIndex << ' ';

    std::cout << "\nfirst: " << indexList.first << '\n';
    std::cout << "Task 2 PASSED: storage structures are built correctly." << std::endl;
    return 0;
}


In [ ]:
from pathlib import Path
import subprocess

result = subprocess.run(
    [
        "g++",
        "-std=c++17",
        "-I", "Source/01.02/cpp",
        "Source/01.02/cpp/practice_build.cpp",
        "Source/01.02/cpp/task2_test.cpp",
        "-o", "Source/01.02/cpp/task2_test",
    ],
    text=True,
    capture_output=True,
)

if result.returncode != 0:
    print("Task 2 编译未通过，请检查 TODO 部分：")
    print(result.stderr)
else:
    run_result = subprocess.run(
        ["Source/01.02/cpp/task2_test"],
        text=True,
        capture_output=True,
    )

    if run_result.returncode != 0:
        print("Task 2 运行未通过，请检查数据结构构建逻辑。")
        print(run_result.stdout)
        print(run_result.stderr)
    else:
        print(run_result.stdout)

---
## 3. 任务三：实现求和算子

这一任务直接对应完整实验中两种求和路径最核心的访问逻辑。

### 3.1 顺序存储求和

顺序存储直接按照物理下标：

```text
0 → 1 → 2 → 3
```

依次读取并累加。

### 3.2 链式存储求和

链式存储不能简单使用 `0,1,2,3` 作为访问顺序，而应：

```text
first → nextIndex → nextIndex → ... → -1
```

每访问一个结点后，必须先读取该结点的 `nextIndex`，才能知道下一步访问哪里。

### 3.3 记录链式访问顺序

除求和外，还需要实现 `getVisitOrder`，把链式遍历过程中真正访问到的**物理下标**依次保存下来。

提示：可以使用 `visited` 记录已经访问的结点数量，避免异常链表导致无限循环。


In [ ]:
%%writefile Source/01.02/cpp/practice_traversal.hpp
#pragma once

#include "practice_types.hpp"

#include <vector>

float sumArrayList(const ArrayList& list);
float sumIndexList(const IndexList& list);
std::vector<int> getVisitOrder(const IndexList& list);


In [ ]:
%%writefile Source/01.02/cpp/practice_traversal.cpp
#include "practice_traversal.hpp"

#include <cstddef>
#include <vector>

float sumArrayList(const ArrayList& list)
{
    float sum = 0.0f;

    // TODO 3A：
    // 按顺序访问 list.data 中的所有元素并完成累加。

    return sum;
}

float sumIndexList(const IndexList& list)
{
    float sum = 0.0f;
    int current = list.first;
    std::size_t visited = 0;

    // TODO 3B：
    // 从 first 开始沿 nextIndex 遍历。
    // 每访问一个结点：
    // 1. 累加当前结点的数据；
    // 2. 更新 current，使其指向下一个物理结点；
    // 3. 更新 visited。
    //
    // 建议的循环停止条件：
    // - current == -1，表示到达链尾；
    // - 或 visited 达到结点总数，避免异常链表造成死循环。

    return sum;
}

std::vector<int> getVisitOrder(const IndexList& list)
{
    std::vector<int> order;
    int current = list.first;
    std::size_t visited = 0;

    // TODO 3C：
    // 按照与 sumIndexList 相同的链式遍历方式，
    // 将每次访问到的物理下标 current 依次保存到 order 中。

    return order;
}


In [ ]:
%%writefile Source/01.02/cpp/task3_test.cpp
#include "practice_build.hpp"
#include "practice_traversal.hpp"

#include <cassert>
#include <cmath>
#include <iostream>
#include <vector>

int main()
{
    const std::vector<float> input = {10.0f, 20.0f, 30.0f, 40.0f};

    const ArrayList arrayList = buildArrayList(input);
    const IndexList indexList = buildIndexList(input);

    const float arraySum = sumArrayList(arrayList);
    const float indexSum = sumIndexList(indexList);
    const std::vector<int> order = getVisitOrder(indexList);

    assert(std::fabs(arraySum - 100.0f) < 1e-6f);
    assert(std::fabs(indexSum - 100.0f) < 1e-6f);
    assert((order == std::vector<int>{0, 2, 3, 1}));

    std::cout << "Sequential access indices: 0 1 2 3\n";

    std::cout << "Linked access indices: ";
    for (int idx : order) std::cout << idx << ' ';
    std::cout << '\n';

    std::cout << "ArrayList sum: " << arraySum << '\n';
    std::cout << "IndexList sum: " << indexSum << '\n';

    std::cout << "Task 3 PASSED: traversal, sum and visit order are correct." << std::endl;
    return 0;
}


In [ ]:
from pathlib import Path
import subprocess

result = subprocess.run(
    [
        "g++",
        "-std=c++17",
        "-I", "Source/01.02/cpp",
        "Source/01.02/cpp/practice_build.cpp",
        "Source/01.02/cpp/practice_traversal.cpp",
        "Source/01.02/cpp/task3_test.cpp",
        "-o", "Source/01.02/cpp/task3_test",
    ],
    text=True,
    capture_output=True,
)

if result.returncode != 0:
    print("Task 3 编译未通过，请检查 TODO 部分：")
    print(result.stderr)
else:
    run_result = subprocess.run(
        ["Source/01.02/cpp/task3_test"],
        text=True,
        capture_output=True,
    )

    if run_result.returncode != 0:
        print("Task 3 运行未通过，请检查遍历、求和或访问顺序实现。")
        print(run_result.stdout)
        print(run_result.stderr)
    else:
        print(run_result.stdout)

---
## 4. 任务四：Ascend C实现求和算子

前 3 个任务使用标准 C++ 理解了两种存储结构。现在把相同的核心访问逻辑迁移到 NPU。

为了避免重新搭建完整工程，本任务已经提供：

- CANN 环境检测；
- Host 侧内存申请、数据搬运和 Kernel 启动；
- CMake 构建配置；
- 运行脚本。

学生只需要补全两个 Ascend C Kernel 中的**遍历与累加部分**。

### 4.1 数据表示的变化

C++ 小型 Demo 中，链式结构使用：

```cpp
ListNode {
    float data;
    int nextIndex;
};
```

在 NPU Kernel 中，为便于通过 `GlobalTensor` 访问，将它拆成两个数组：

```text
nodeData[]   ：保存结点数据
nextIndex[]  ：保存后继物理下标
```

但链式遍历思想不变：

```text
current = first
      ↓
读取 nodeData[current]
      ↓
current = nextIndex[current]
      ↓
继续访问
```


### 4.2 顺序存储 Ascend C Kernel

请补全 `TODO 4A`。

提示：

- `dataGm.GetValue(i)` 可以读取第 `i` 个元素；
- 顺序存储直接按 `0 → 1 → ... → n-1` 遍历；
- 本任务只要求完成遍历和累加，不需要修改 `GlobalTensor` 初始化和结果写回。


In [ ]:
%%writefile Source/01.02/ascend_ops/op_kernel/array_sum_practice.cpp
#include "kernel_operator.h"

using namespace AscendC;

extern "C" __global__ __aicore__ void array_sum_practice(
    GM_ADDR data,
    GM_ADDR out,
    uint32_t n)
{
    InitSocState();

    if (GetBlockIdx() != 0) {
        return;
    }

    GlobalTensor<float> dataGm;
    GlobalTensor<float> outGm;
    dataGm.SetGlobalBuffer(reinterpret_cast<__gm__ float *>(data), n);
    outGm.SetGlobalBuffer(reinterpret_cast<__gm__ float *>(out), 1);

    float sum = 0.0f;

    // TODO 4A：
    // 按物理下标从 0 到 n-1 顺序读取 dataGm，并完成累加。

    outGm.SetValue(0, sum);
    DataCacheCleanAndInvalid<float, CacheLine::ENTIRE_DATA_CACHE,
                             DcciDst::CACHELINE_OUT>(outGm);
}


### 4.3 链式存储 Ascend C Kernel

请补全 `TODO 4B`。

提示：

- `dataGm.GetValue(current)` 读取当前结点数据；
- `nextGm.GetValue(current)` 得到下一个结点的物理下标；
- `first` 是链式遍历起点；
- `current == -1` 表示到达链尾；
- 建议保留访问次数上界，避免异常链表导致死循环。


In [ ]:
%%writefile Source/01.02/ascend_ops/op_kernel/index_sum_practice.cpp
#include "kernel_operator.h"

using namespace AscendC;

extern "C" __global__ __aicore__ void index_sum_practice(
    GM_ADDR nodeData,
    GM_ADDR nextIndex,
    GM_ADDR out,
    uint32_t n,
    int32_t first)
{
    InitSocState();

    if (GetBlockIdx() != 0) {
        return;
    }

    GlobalTensor<float> dataGm;
    GlobalTensor<int32_t> nextGm;
    GlobalTensor<float> outGm;

    dataGm.SetGlobalBuffer(reinterpret_cast<__gm__ float *>(nodeData), n);
    nextGm.SetGlobalBuffer(reinterpret_cast<__gm__ int32_t *>(nextIndex), n);
    outGm.SetGlobalBuffer(reinterpret_cast<__gm__ float *>(out), 1);

    float sum = 0.0f;
    int32_t current = first;
    uint32_t visited = 0;

    // TODO 4B：
    // 从 first 开始沿 nextIndex 遍历。
    // 每访问一个结点，完成：
    // 1. 读取当前结点的数据并累加；
    // 2. 根据 nextGm 更新 current；
    // 3. 更新 visited。
    //
    // 循环条件应同时考虑：
    // - current 是否仍为有效物理下标；
    // - visited 是否小于 n。

    outGm.SetValue(0, sum);
    DataCacheCleanAndInvalid<float, CacheLine::ENTIRE_DATA_CACHE,
                             DcciDst::CACHELINE_OUT>(outGm);
}


### 4.4 NPU 工程辅助代码

下面的 Host、CMake 和运行脚本已经提供，**不需要修改**。

Host 端使用与前面 C++ Demo 相同的小规模数据：

```text
顺序存储：
10 20 30 40

链式存储的物理数据：
10 40 20 30

nextIndex：
2 -1 3 1

first：
0
```

两个 Kernel 的正确结果都应为 `100`。


In [ ]:
%%writefile Source/01.02/ascend_ops/host_launch/practice_npu_main.cpp
#include <acl/acl.h>
#include "aclrtlaunch_array_sum_practice.h"
#include "aclrtlaunch_index_sum_practice.h"

#include <cmath>
#include <cstdint>
#include <iostream>
#include <stdexcept>
#include <string>
#include <vector>

#define ACL_CHECK(expr)                                                                    \
    do {                                                                                   \
        aclError _ret = (expr);                                                            \
        if (_ret != ACL_SUCCESS) {                                                         \
            throw std::runtime_error(std::string("[ACL ERROR] ") + #expr +               \
                                     " failed, ret=" + std::to_string(_ret));             \
        }                                                                                  \
    } while (0)

namespace {

constexpr uint32_t N = 4;
constexpr float REF = 100.0f;

bool close_enough(float value)
{
    return std::fabs(value - REF) < 1e-5f;
}

}  // namespace

int main()
{
    const std::vector<float> arrayData = {10.0f, 20.0f, 30.0f, 40.0f};
    const std::vector<float> nodeData = {10.0f, 40.0f, 20.0f, 30.0f};
    const std::vector<int32_t> nextIndex = {2, -1, 3, 1};
    const int32_t first = 0;

    aclrtStream stream = nullptr;
    float *arrayDev = nullptr;
    float *nodeDev = nullptr;
    int32_t *nextDev = nullptr;
    float *outDev = nullptr;

    bool aclInited = false;
    bool deviceSet = false;

    auto cleanup = [&]() {
        if (arrayDev) (void)aclrtFree(arrayDev);
        if (nodeDev) (void)aclrtFree(nodeDev);
        if (nextDev) (void)aclrtFree(nextDev);
        if (outDev) (void)aclrtFree(outDev);
        if (stream) (void)aclrtDestroyStream(stream);
        if (deviceSet) (void)aclrtResetDevice(0);
        if (aclInited) (void)aclFinalize();
    };

    try {
        ACL_CHECK(aclInit(nullptr));
        aclInited = true;

        ACL_CHECK(aclrtSetDevice(0));
        deviceSet = true;

        ACL_CHECK(aclrtCreateStream(&stream));

        const std::size_t dataBytes = N * sizeof(float);
        const std::size_t nextBytes = N * sizeof(int32_t);

        ACL_CHECK(aclrtMalloc(reinterpret_cast<void **>(&arrayDev),
                             dataBytes, ACL_MEM_MALLOC_HUGE_FIRST));
        ACL_CHECK(aclrtMalloc(reinterpret_cast<void **>(&nodeDev),
                             dataBytes, ACL_MEM_MALLOC_HUGE_FIRST));
        ACL_CHECK(aclrtMalloc(reinterpret_cast<void **>(&nextDev),
                             nextBytes, ACL_MEM_MALLOC_HUGE_FIRST));
        ACL_CHECK(aclrtMalloc(reinterpret_cast<void **>(&outDev),
                             sizeof(float), ACL_MEM_MALLOC_HUGE_FIRST));

        ACL_CHECK(aclrtMemcpy(arrayDev, dataBytes,
                              arrayData.data(), dataBytes,
                              ACL_MEMCPY_HOST_TO_DEVICE));
        ACL_CHECK(aclrtMemcpy(nodeDev, dataBytes,
                              nodeData.data(), dataBytes,
                              ACL_MEMCPY_HOST_TO_DEVICE));
        ACL_CHECK(aclrtMemcpy(nextDev, nextBytes,
                              nextIndex.data(), nextBytes,
                              ACL_MEMCPY_HOST_TO_DEVICE));

        float zero = 0.0f;
        float arrayResult = 0.0f;
        float indexResult = 0.0f;

        ACL_CHECK(aclrtMemcpy(outDev, sizeof(float), &zero, sizeof(float),
                              ACL_MEMCPY_HOST_TO_DEVICE));
        ACLRT_LAUNCH_KERNEL(array_sum_practice)(1, stream, arrayDev, outDev, N);
        ACL_CHECK(aclrtSynchronizeStream(stream));
        ACL_CHECK(aclrtMemcpy(&arrayResult, sizeof(float), outDev, sizeof(float),
                              ACL_MEMCPY_DEVICE_TO_HOST));

        ACL_CHECK(aclrtMemcpy(outDev, sizeof(float), &zero, sizeof(float),
                              ACL_MEMCPY_HOST_TO_DEVICE));
        ACLRT_LAUNCH_KERNEL(index_sum_practice)(1, stream,
                                                nodeDev, nextDev, outDev,
                                                N, first);
        ACL_CHECK(aclrtSynchronizeStream(stream));
        ACL_CHECK(aclrtMemcpy(&indexResult, sizeof(float), outDev, sizeof(float),
                              ACL_MEMCPY_DEVICE_TO_HOST));

        const bool arrayPass = close_enough(arrayResult);
        const bool indexPass = close_enough(indexResult);

        std::cout << "array_sum_practice: result=" << arrayResult
                  << " status=" << (arrayPass ? "PASS" : "FAIL") << '\n';
        std::cout << "index_sum_practice: result=" << indexResult
                  << " status=" << (indexPass ? "PASS" : "FAIL") << '\n';

        const bool allPass = arrayPass && indexPass;
        if (allPass) {
            std::cout << "Task 4 PASSED: Ascend C traversal logic is correct." << std::endl;
        }

        cleanup();
        return allPass ? 0 : 2;
    } catch (const std::exception& e) {
        cleanup();
        std::cerr << "[error] " << e.what() << std::endl;
        return 1;
    }
}


In [ ]:
%%writefile Source/01.02/CMakeLists.txt
cmake_minimum_required(VERSION 3.16)
project(storage_sum_practice LANGUAGES CXX)

set(CMAKE_CXX_STANDARD 17)
set(CMAKE_CXX_STANDARD_REQUIRED ON)
set(CMAKE_CXX_EXTENSIONS OFF)

set(RUN_MODE "npu" CACHE STRING "Ascend C run mode")
set(SOC_VERSION "ascend910b3" CACHE STRING "Ascend SOC version")
set(ASCEND_CANN_PATH "$ENV{ASCEND_INSTALL_PATH}" CACHE PATH "CANN installation path")
set(ASCEND_CANN_PACKAGE_PATH "${ASCEND_CANN_PATH}" CACHE PATH "CANN package path" FORCE)
set(CMAKE_INSTALL_PREFIX "${CMAKE_BINARY_DIR}/out" CACHE PATH "Ascend C install output" FORCE)

if(EXISTS "${ASCEND_CANN_PACKAGE_PATH}/tools/tikcpp/ascendc_kernel_cmake/ascendc.cmake")
  set(ASCENDC_CMAKE_FILE "${ASCEND_CANN_PACKAGE_PATH}/tools/tikcpp/ascendc_kernel_cmake/ascendc.cmake")
elseif(EXISTS "${ASCEND_CANN_PACKAGE_PATH}/compiler/tikcpp/ascendc_kernel_cmake/ascendc.cmake")
  set(ASCENDC_CMAKE_FILE "${ASCEND_CANN_PACKAGE_PATH}/compiler/tikcpp/ascendc_kernel_cmake/ascendc.cmake")
elseif(EXISTS "${ASCEND_CANN_PACKAGE_PATH}/tikcpp/ascendc_kernel_cmake/ascendc.cmake")
  set(ASCENDC_CMAKE_FILE "${ASCEND_CANN_PACKAGE_PATH}/tikcpp/ascendc_kernel_cmake/ascendc.cmake")
else()
  message(FATAL_ERROR "Cannot find ascendc.cmake under ${ASCEND_CANN_PACKAGE_PATH}")
endif()

include("${ASCENDC_CMAKE_FILE}")

ascendc_library(practice_array_kernel STATIC
    ascend_ops/op_kernel/array_sum_practice.cpp
)
ascendc_compile_definitions(practice_array_kernel PRIVATE -DASCENDC_DUMP=0)

ascendc_library(practice_index_kernel STATIC
    ascend_ops/op_kernel/index_sum_practice.cpp
)
ascendc_compile_definitions(practice_index_kernel PRIVATE -DASCENDC_DUMP=0)

add_executable(storage_sum_practice_npu
    ascend_ops/host_launch/practice_npu_main.cpp
)

target_include_directories(storage_sum_practice_npu PRIVATE
    ${ASCEND_CANN_PACKAGE_PATH}/include
    ${ASCEND_CANN_PACKAGE_PATH}/include/external
    ${ASCEND_CANN_PACKAGE_PATH}/runtime/include
    ${ASCEND_CANN_PACKAGE_PATH}/compiler/tikcpp
    ${ASCEND_CANN_PACKAGE_PATH}/compiler/tikcpp/tikcfw
    ${CMAKE_INSTALL_PREFIX}/include/practice_array_kernel
    ${CMAKE_INSTALL_PREFIX}/include/practice_index_kernel
    ${CMAKE_BINARY_DIR}/out/include/practice_array_kernel
    ${CMAKE_BINARY_DIR}/out/include/practice_index_kernel
)

target_link_directories(storage_sum_practice_npu PRIVATE
    ${ASCEND_CANN_PACKAGE_PATH}/lib64
    ${ASCEND_CANN_PACKAGE_PATH}/runtime/lib64/stub
    ${ASCEND_CANN_PACKAGE_PATH}/runtime/lib64
    ${ASCEND_CANN_PACKAGE_PATH}/acllib/lib64
    ${ASCEND_CANN_PACKAGE_PATH}/aarch64-linux/devlib
    ${ASCEND_CANN_PACKAGE_PATH}/x86_64-linux/devlib
)

target_compile_options(storage_sum_practice_npu PRIVATE
    $<$<CXX_COMPILER_ID:GNU,Clang>:-O2 -Wall -Wextra -Wno-deprecated-declarations>
)

target_link_libraries(storage_sum_practice_npu PRIVATE
    practice_array_kernel
    practice_index_kernel
    ascendcl
    platform
    ascendalog
    c_sec
    dl
)

add_dependencies(storage_sum_practice_npu
    practice_array_kernel
    practice_index_kernel
)


In [ ]:
%%writefile Source/01.02/scripts/run_npu.sh
#!/usr/bin/env bash
set -euo pipefail

SCRIPT_DIR=$(cd "$(dirname "$0")/.." && pwd)
BUILD_DIR="${SCRIPT_DIR}/build_ascend"
ASCEND_INSTALL_PATH_DEFAULT="/usr/local/Ascend/ascend-toolkit/latest"

is_cann_install_path() {
  local path="$1"
  [[ -f "${path}/tikcpp/ascendc_kernel_cmake/ascendc.cmake" ||
     -f "${path}/tools/tikcpp/ascendc_kernel_cmake/ascendc.cmake" ||
     -f "${path}/compiler/tikcpp/ascendc_kernel_cmake/ascendc.cmake" ]]
}

detect_cann_install_path() {
  local arch candidate
  arch=$(uname -m)

  if [[ -n "${ASCEND_HOME_PATH:-}" ]]; then
    candidate="${ASCEND_HOME_PATH}/${arch}-linux"
    if is_cann_install_path "${candidate}"; then
      echo "${candidate}"
      return 0
    fi
  fi

  for candidate in \
    /opt/conda/Ascend/cann-*/"${arch}-linux" \
    /usr/local/Ascend/ascend-toolkit/latest \
    /usr/local/Ascend/ascend-toolkit/*/"${arch}-linux"; do
    if is_cann_install_path "${candidate}"; then
      echo "${candidate}"
      return 0
    fi
  done
  return 1
}

detect_soc_version() {
  local raw_name normalized
  if ! command -v npu-smi >/dev/null 2>&1; then
    return 1
  fi

  raw_name=$(npu-smi info 2>/dev/null \
    | sed -nE 's/^\|[[:space:]]*[0-9]+[[:space:]]+([^|]+)\|.*/\1/p' \
    | head -n 1 || true)
  normalized=$(echo "${raw_name}" \
    | tr '[:upper:]' '[:lower:]' \
    | tr -cd '[:alnum:]')

  case "${normalized}" in
    ascend910*|ascend310*) echo "${normalized}" ;;
    910*|310*) echo "ascend${normalized}" ;;
    *) return 1 ;;
  esac
}

if [[ -n "${ASCEND_INSTALL_PATH:-}" ]] && is_cann_install_path "${ASCEND_INSTALL_PATH}"; then
  :
elif detected_path=$(detect_cann_install_path); then
  ASCEND_INSTALL_PATH="${detected_path}"
else
  ASCEND_INSTALL_PATH="${ASCEND_INSTALL_PATH_DEFAULT}"
fi

if [[ ! -d "${ASCEND_INSTALL_PATH}" ]]; then
  echo "ASCEND_INSTALL_PATH does not exist: ${ASCEND_INSTALL_PATH}" >&2
  exit 1
fi

if detected_soc=$(detect_soc_version); then
  SOC_VERSION="${detected_soc}"
elif [[ -n "${SOC_VERSION:-}" ]]; then
  SOC_VERSION="${SOC_VERSION}"
else
  echo "Cannot detect SOC_VERSION. Set SOC_VERSION manually." >&2
  exit 1
fi

if [[ -f "${ASCEND_INSTALL_PATH}/set_env.sh" ]]; then
  # shellcheck disable=SC1090
  source "${ASCEND_INSTALL_PATH}/set_env.sh"
fi

echo "[INFO] ASCEND_INSTALL_PATH=${ASCEND_INSTALL_PATH}"
echo "[INFO] SOC_VERSION=${SOC_VERSION}"

rm -rf "${BUILD_DIR}"
mkdir -p "${BUILD_DIR}"

# 兼容 CANNLab 中部分新版 CANN 包目录布局。
ASCEND_CMAKE_PACKAGE_PATH="${ASCEND_INSTALL_PATH}"
ASCENDC_CMAKE_NEW_LAYOUT="${ASCEND_INSTALL_PATH}/tikcpp/ascendc_kernel_cmake/ascendc.cmake"
BISHENG_REAL="${ASCEND_INSTALL_PATH}/ccec_compiler/bin/bisheng"
BISHENG_EXPECTED="${ASCEND_INSTALL_PATH}/ascendc_devkit/ccec_compiler/bin/bisheng"

if [[ -f "${ASCENDC_CMAKE_NEW_LAYOUT}" && -x "${BISHENG_REAL}" && ! -x "${BISHENG_EXPECTED}" ]]; then
  ASCEND_CMAKE_PACKAGE_PATH="${BUILD_DIR}/cann_package_compat"
  mkdir -p "${ASCEND_CMAKE_PACKAGE_PATH}/ascendc_devkit"

  for entry in "${ASCEND_INSTALL_PATH}"/*; do
    [[ -e "${entry}" ]] || continue
    name=$(basename "${entry}")
    [[ "${name}" == "ascendc_devkit" ]] && continue
    ln -sfn "${entry}" "${ASCEND_CMAKE_PACKAGE_PATH}/${name}"
  done

  if [[ -d "${ASCEND_INSTALL_PATH}/ascendc_devkit" ]]; then
    for entry in "${ASCEND_INSTALL_PATH}/ascendc_devkit"/*; do
      [[ -e "${entry}" ]] || continue
      name=$(basename "${entry}")
      [[ "${name}" == "ccec_compiler" ]] && continue
      ln -sfn "${entry}" "${ASCEND_CMAKE_PACKAGE_PATH}/ascendc_devkit/${name}"
    done
  fi

  for component in asc tikcpp ccec_compiler; do
    if [[ -e "${ASCEND_INSTALL_PATH}/${component}" ]]; then
      ln -sfn "${ASCEND_INSTALL_PATH}/${component}" \
        "${ASCEND_CMAKE_PACKAGE_PATH}/ascendc_devkit/${component}"
    fi
  done
  echo "[INFO] Using CANN package compatibility view: ${ASCEND_CMAKE_PACKAGE_PATH}"
fi

export ASCEND_INSTALL_PATH
export ASCEND_CANN_PACKAGE_PATH="${ASCEND_CMAKE_PACKAGE_PATH}"
export SOC_VERSION

cd "${BUILD_DIR}"

cmake "${SCRIPT_DIR}" \
  -DCMAKE_BUILD_TYPE=Release \
  -DASCEND_CANN_PATH="${ASCEND_CMAKE_PACKAGE_PATH}" \
  -DASCEND_CANN_PACKAGE_PATH="${ASCEND_CMAKE_PACKAGE_PATH}" \
  -DSOC_VERSION="${SOC_VERSION}" \
  -DRUN_MODE=npu

cmake --build . -j

"${BUILD_DIR}/storage_sum_practice_npu"


In [ ]:
!chmod +x Source/01.02/scripts/run_npu.sh
print("NPU auxiliary files are ready.")


### 4.5 编译并运行 NPU 实践

完成 `TODO 4A` 和 `TODO 4B` 后运行下面的测试单元。

该单元会先检查 Kernel 中是否仍然存在 TODO；全部补全后才执行 CMake 构建和 NPU 运行。

预期结果：

```text
array_sum_practice: result=100 status=PASS
index_sum_practice: result=100 status=PASS
Task 4 PASSED: Ascend C traversal logic is correct.
```


In [ ]:
import subprocess

result = subprocess.run(
    [
        "bash",
        "Source/01.02/scripts/run_npu.sh",
    ],
    text=True,
    capture_output=True,
)

if result.returncode != 0:
    print("Task 4 构建或运行未通过，请检查 Ascend C Kernel 中的 TODO 部分：")
    print(result.stdout)
    print(result.stderr)
else:
    print(result.stdout)

---
## 5. 实践总结

完成四项任务后，应能够把两种存储结构的实现关系对应起来：

```text
顺序存储（C++）
ArrayList::data[i]
        ↓
Ascend C
dataGm.GetValue(i)
```

```text
链式存储（C++）
current = node.nextIndex
        ↓
Ascend C
current = nextGm.GetValue(current)
```

本实践中的两种实现处理的是同一组逻辑数据，因此求和结果应相同；不同之处在于实际访问路径：

```text
顺序存储：
0 → 1 → 2 → 3

链式存储：
0 → 2 → 3 → 1
```
